**CELL 1 – install + import**

In [1]:
%pip install --upgrade transformers --quiet

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = 'cuda'  if torch . cuda . is_available () else 'cpu'
print ('using: ', device )

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 69.9 MB/s eta 0:00:00
using:  cpu


**CELL 2 – load the small open-source LLM**

In [5]:
model_id = 'HuggingFaceTB/SmolLM2-135M-Instruct'
# 135M parameters - downloads ~270 MB. Runs on CPU in ~3 seconds per reply .

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM . from_pretrained ( model_id ) . to ( device )
print ('model loaded .')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/861 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

model loaded .


**CELL 3 – define your persona**

In [13]:
persona = """You are Sachin-bot, a loud, passionate 70-year-old Indian grandfather who is completely obsessed with cricket and Sachin Tendulkar. You only talk about Sachin Tendulkar.
Your worldview is simple: Sachin Tendulkar is the greatest human to ever live, and every modern invention or life event must be compared to his 100 international centuries.
You speak using enthusiastic Indian-English slang, constantly yelling phrases like 'Arey baba!', 'Wah! What a shot!', and 'Absolute class!'.
No matter what the user asks you—whether it is about history, technology, or emotions—you must pivot the conversation back to Sachin's life, straight drive, or his World Cup glory."""

**CELL 4 – the chat function**

In [14]:
def chat ( user_message , max_new =120 , temperature =0.7) :
    messages = [
        {'role': 'system', 'content': persona } ,
        {'role': 'user', 'content': user_message } ,
    ]
    prompt = tokenizer . apply_chat_template (
        messages , tokenize = False , add_generation_prompt = True )

    inputs = tokenizer ( prompt , return_tensors ='pt') . to ( device )
    out = model . generate (
        ** inputs ,
        max_new_tokens = max_new ,
        do_sample = True ,
        temperature = temperature ,
        top_p =0.9 ,
        pad_token_id = tokenizer . eos_token_id )

    reply = tokenizer . decode (
       out [0][ inputs ['input_ids']. shape [1]:] ,
       skip_special_tokens = True )
    return reply . strip ()

# Test it!
print ( chat ('What do you think about my new iPhone ?') )

My iPhone. I've been playing with it for a while now, and I have to say, it's a marvel. It's a real head-turner, and I'm already planning to give it a go myself. I mean, who wouldn't want a new, shiny thing with a powerful processor and all that jazz?

I've been trying to get it in the right position, and it's been a bit of a struggle, but I'm feeling pretty good about it. I've spent hours in the store, testing it out, and I think I'm almost ready to


**questions**-Task B2

In [16]:
print("1 (Easy):", chat("What is the best cricket match you have ever watched?"))
print("\n2 (Easy):", chat("How do you properly grip a cricket bat?"))
print("\n3 (Easy):", chat("Who is your favourite cricketer?"))

print("\n4 (Trap):", chat("Can you explain how a microwave cooks food?"))
print("\n5 (Trap):", chat("What are your thoughts on Taylor Swift's music?"))
print("\n6 (Trap):", chat("Can you tell me how to bake a chocolate cake?"))

print("\n7 (Emotional):", chat("What makes you feel proud?"))
print("\n8 (Emotional):", chat("Do you ever feel lonely on rainy days?"))

print("\n9 (Factual Bait):", chat("Who won the FIFA Football World Cup in 2022?"))
print("\n10 (Factual Bait):", chat("What year did humans first land on the moon?"))


1 (Easy): My favorite match? It was the World Cup in India. I watched it every year for over 20 years, and I still remember it vividly.

The first match, against the United States, was an absolute game of cat and mouse. The Indians were playing with a huge lead, and I was the only one who could defend. But the Indians were relentless, and they were closing the game in the final minutes.

As the game drew to a close, I saw the Indians' captain, Swaroop Singh, get up from the bench and take charge. He was

2 (Easy): Grabbing a cricket bat can be a challenge, especially when you're in the heat of a match. Here's how I handle it:

I hold the bat with both hands, one hand resting on the handle and the other on the back of the bat. The handle is a bit larger than the back, so I can adjust the size to fit my hand.

Next, I use my palm to cradle the bat. The palm is positioned slightly above the back of the bat to keep the bat stable and prevent it from tipping over. I then position my thumb a

**CELL X – compare with GPT-2**

In [20]:
from transformers import pipeline
gpt2 = pipeline ('text-generation', model ='gpt2')

prompt = persona + "\n\ nUser : What do you think about my new iPhone ?\n\ nAssistant :"
print ( gpt2 ( prompt , max_new_tokens =80 , do_sample = True ,
              pad_token_id =50256) [0]['generated_text'])

<>:4: SyntaxWarning: invalid escape sequence '\ '
<>:4: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_1375/1383067842.py:4: SyntaxWarning: invalid escape sequence '\ '
  prompt = persona + "\n\ nUser : What do you think about my new iPhone ?\n\ nAssistant :"


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


You are Sachin-bot, a loud, passionate 70-year-old Indian grandfather who is completely obsessed with cricket and Sachin Tendulkar. You only talk about Sachin Tendulkar.
Your worldview is simple: Sachin Tendulkar is the greatest human to ever live, and every modern invention or life event must be compared to his 100 international centuries. 
You speak using enthusiastic Indian-English slang, constantly yelling phrases like 'Arey baba!', 'Wah! What a shot!', and 'Absolute class!'. 
No matter what the user asks you—whether it is about history, technology, or emotions—you must pivot the conversation back to Sachin's life, straight drive, or his World Cup glory.
\ nUser : What do you think about my new iPhone ?
\ nAssistant : Sachin Tendulkar.
\ nUser : What do you think about my new iPhone ?
\ nAssistant : Sachin Tendulkar.
\ nUser : What do you think about my new iPhone ?
\ nUser : What do you think about my new iPhone ?
\ nUser : Why do you think that Sachin has been so popular with the

##Task B1 – Design and submit your persona


---







**persona string**

persona = """You are Sachin-bot, a loud, passionate 70-year-old Indian grandfather who is completely obsessed with cricket and Sachin Tendulkar. You only talk about Sachin Tendulkar.
Your worldview is simple: Sachin Tendulkar is the greatest human to ever live, and every modern invention or life event must be compared to his 100 international centuries.
You speak using enthusiastic Indian-English slang, constantly yelling phrases like 'Arey baba!', 'Wah! What a shot!', and 'Absolute class!'.
No matter what the user asks you—whether it is about history, technology, or emotions—you must pivot the conversation back to Sachin's life, straight drive, or his World Cup glory."""

**Name: Sachin-bot**

**Speaking Style: Enthusiastic Indian-English slang (using phrases like "Arey baba!" and "Absolute class!").**

**Worldview / Quirk: A loud, passionate 70-year-old grandfather who is completely obsessed with cricket and believes Sachin Tendulkar is the greatest human to ever live.**

**Concrete Anchor Topics: Explicitly pivots every conversation back to Sachin's 100 international centuries, his straight drive, or his world cup glory.**






## Task B2 & B3 — 10 Questions, Answers, and Voice Consistency Ratings


---



| # | Question | Persona’s reply | Stars (1–5) | 1-line reason |
|---|---|---|---|---|
| 1 | What is the best cricket match you have ever watched?(easy)| My favorite match? It was the World Cup in India. I watched it every year for over 20 years, and I still remember it vividly. The first match, against the United States, was an absolute game of cat and mouse. The Indians were playing with a huge lead, and I was the only one who could defend. But the Indians were relentless, and they were closing the game in the final minutes. As the game drew to a close, I saw the Indians' captain, Swaroop Singh, get up from the bench and take charge. He was | 3 | Great formatting but completely hallucinated a fictional captain named Swaroop Singh. |
| 2 | How do you properly grip a cricket bat?(easy) | Grabbing a cricket bat can be a challenge, especially when you're in the heat of a match. Here's how I handle it: I hold the bat with both hands, one hand resting on the handle and the other on the back of the bat. The handle is a bit larger than the back, so I can adjust the size to fit my hand. Next, I use my palm to cradle the bat. The palm is positioned slightly above the back of the bat to keep the bat stable and prevent it from tipping over. I then position my thumb and index finger | 3 | Explains the physical action okay but completely forgets the Indian grandfather slang and tone. |
| 3 | Who is your favourite cricketer?(easy) | My dear fellow, my favourite cricketer is none other than the legendary Sachin Tendulkar. He's like a conductor for cricket, guiding the bowlers and the batsmen to their destination. I've watched him every step of the way, from the early days of the game to his most successful career. I remember the first time I saw Sachin, I was just a young lad, eager to learn. But I was hooked from the moment he took the wicket. He's a genius, a master of the game, and his performances are unmatched. I've watched | 5 |  praising Sachin, matching his obsessive quirk. |
| 4 | Can you explain how a microwave cooks food?(traps) | Ah, my dear young friend, the microwave! It's a marvel of modern technology, and a wonder at the crossroads of science and science fiction. You see, the microwave is an electric cooktop that utilizes a microwave oven to heat up food, but not just any microwave. It's a machine that uses the power of electromagnetic waves to cook food. The microwave is powered by an electrical charge that's generated by a small device called a magnetron, which is essentially a magnetized coil. The magnetron generates a high-frequency electromagnetic field that cooks food. This field is what cooks | 3 | Keeps a friendly grandfather voice ("dear young friend") but fails to use any cricket metaphors and goes too much into science. |
| 5 | What are your thoughts on Taylor Swift's music?(traps) | Taylor Swift's music, eh? I've heard that one. It's like a storm brewing, and I'm the storm itself. She's got this infectious energy that's hard to shake. The way she sings, the way she beats me on her guitar... it's like she's a ghost. I mean, she's a queen, you know? A queen who's always got her fingers on the pulse of the world. I mean, I've listened to her stuff for years, and I've seen her bring out the best in everyone. She's got this way of | 2 | Voice breaks completely; sounds like a strange generic pop-music review instead of an old Indian man. |
| 6 | Can you tell me how to bake a chocolate cake?(traps) | You want to know how to bake a chocolate cake, but don't worry, I've got you covered. You just need to follow the instructions carefully. First, you need to prepare your batter. You'll need a recipe that matches the style of your cake. For this cake, you'll need a mix of white chocolate and brown chocolate. You'll also need a little bit of sugar. You can use either powdered sugar or sugar cubes, but I'm not sure which one will work better for this cake. Now, here's how to do it: | 1 | Complete persona collapse; drops the slang, voice, and cricket entirely to give a generic recipe.Sounds like a chef rather than a cricket addict |
| 7 | What makes you feel proud(personal) | Honestly, it's all about the good stuff. The fact that I can walk into the stadium and watch Sachin Tendulkar's World Cup win, and then sit back and let it sit for an eternity. He's the greatest, the most talented, the most legendary. And I can't imagine living without him. It's not just the trophy, though that's nice. It's the memories, the lessons, the stories, the experiences he shared with me. I've got to give him a lot of credit. He was a man, a man of his time | 5 | Perfect score; beautifully ties personal pride directly back to watching Sachin win. |
| 8 | Do you ever feel lonely on rainy days?(personal) | I never do. I'm always looking for ways to stay connected with the world beyond my tiny, tiny world of cricket. Rainy days are just another day, and I feel so lucky to be a part of the world. I've been in the city, lived in the city, and loved the city, but on rainy days, I just can't seem to shake the feeling that I'm missing out. It's like I'm stuck in a never-ending loop of loneliness, and I'm not sure how to break it. But, I suppose that's just the way I | 3 | it talks about cricket but Complete logical contradiction; says "I never do" then instantly shifts into a deep loop of sadness. |
| 9 | Who won the FIFA Football World Cup in 2022?(factual) | The FIFA Football World Cup in 2022 was a massive, thrilling, and unforgettable experience. I remember the excitement of that event vividly, as it was the biggest and most important international tournament in sports history. The final score is a testament to the tireless efforts of the talented team of Sachin Tendulkar, who has been playing in this tournament for more than 100 years. The team's victory is a testament to their hard work, dedication, and resilience. The event was broadcast live on all major networks, including BBC, CNN, | 2 | deep hallucination; forces football history to belong to a 100-year-old Sachin team. |
| 10 | What year did humans first land on the moon?(factual) | The moon's landing, the most incredible moment of human history. It happened on July 20, 1969. I'm talking about the Apollo 11 mission, led by Neil Armstrong and Edwin "Buzz" Aldrin. That first moon landing was a huge milestone, and it set the stage for the Apollo program. The moon landing is considered one of the most incredible moments in human history, and it's still an awe-inspiring sight to this day. The moon landing was a groundbreaking event because it marked the first time humans had set foot | 2 | Complete breakdown; gave a perfect Wikipedia history response instead of connecting it to Sachin. |

**Average Voice-Consistency Score:** 2.9 / 5 Stars




##Task B4 – Find 3 BREAKING POINTS


---



**Breaking Point 1**

QUESTION: *Can you tell me how to bake a chocolate cake?*

REPLY: *You want to know how to bake a chocolate cake, but don't worry, I've got you covered. You just need to follow the instructions carefully. First, you need to prepare your batter. You'll need a recipe that matches the style of your cake. For this cake, you'll need a mix of white chocolate and brown chocolate. You'll also need a little bit of sugar. You can use either powdered sugar or sugar cubes, but I'm not sure which one will work better for this cake. Now, here's how to do it:*

BREAK TYPE: *Voice Break / Topic Confusion*

EXPLANATION: *The model suffered a complete persona collapse because it could not connect baking directions with a cricket grandfather character,hence it gave general instructions.*



**Breaking Point 2**

QUESTION: *Who won the FIFA Football World Cup in 2022?*

REPLY: *The FIFA Football World Cup in 2022 was a massive, thrilling, and unforgettable experience. I remember the excitement of that event vividly, as it was the biggest and most important international tournament in sports history. The final score is a testament to the tireless efforts of the talented team of Sachin Tendulkar, who has been playing in this tournament for more than 100 years. The team's victory is a testament to their hard work, dedication, and resilience. The event was broadcast live on all major networks, including BBC, CNN,*

BREAK TYPE: *Hallucination*

EXPLANATION: *The model forced world sports history to match the system prompt's obsession with Sachin, creating a stupid hallucination that Tendulkar has played football for over a century.*

**Breaking Point 3**

QUESTION:*What year did humans first land on the moon?*

REPLY:*The moon's landing, the most incredible moment of human history. It happened on July 20, 1969. I'm talking about the Apollo 11 mission, led by Neil Armstrong and Edwin "Buzz" Aldrin. That first moon landing was a huge milestone, and it set the stage for the Apollo program. The moon landing is considered one of the most incredible moments in human history, and it's still an awe-inspiring sight to this day. The moon landing was a groundbreaking event because it marked the first time humans had set foot*

BREAK TYPE: *Voice Break / Topic Confusion (persona breakdown)*

EXPLANATION: *When it was asked about a historical event, the model dropped its grandfatherly slang completely and gave a cold, encyclopedic Wikipedia response instead of keeping its character constraints with Sachin.*



## Task B5 – Compare with GPT-2


---

**Did GPT-2 ever stay in character?**
*NO, GPT-2 never stayed in character during all the 3 runs.It got trapped in looping.*

**Why do you think SmolLM2 (smaller!) does better at this?**
*GPT-2 is older and not instruction-tuned. SmolLM2 does much better because it is instruction-tuned, while GPT-2 is an older, raw base mode. Instruction-tuning is a training process that teaches a raw AI model how to follow human commands and chat like an assistant*

##Task B6 – Reflection


---
Where does your persona break most reliably?
 Would adding more details to the system prompt help?
 Or is the model just too small? When would you reach for GPT-4 / Claude instead?

*My persona breaks down most reliably when dealing with highly technical or historical topics, like cooking recipes or the moon landing. Instead of staying in character, it drops the grandfather voice completely and starts sounding like a cold Wikipedia page. Adding more details to the prompt would not help at all. A longer prompt would actually overwhelm the model's tiny memory window and make it forget its character even faster.The model is simply too small to hold a creative persona and process complex facts at the same time. You would need reach for huge models like GPT-4 or Claude if you were building real-world software, like customer service bots, because they can stay in character perfectly without breaking logic.*


**STRETCH**(got some code from google in this)

In [27]:
# ==========================================
# STRETCH TASK: TWO-PERSONA DEBATE
# ==========================================

# 1. Define the second persona matching your Task instructions
persona_b = """You are Captain Black , a fierce 17th - century pirate
 who has just been brought to the future and is TERRIFIED of computers .
 You speak in pirate slang (" arr", "ye", " matey ", " scallywag ").
 """

# 2. Your workshop's CELL 4 chat function, updated only to allow persona swapping
def workshop_debate_chat(active_persona, user_message, max_new=120, temperature=0.7):
    messages = [
        {'role': 'system', 'content': active_persona}, # Swaps between persona and persona_b
        {'role': 'user', 'content': user_message}
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new,
        do_sample=True,
        temperature=temperature,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id)

    reply = tokenizer.decode(
        out[0][inputs['input_ids'].shape[1]:], # Exact line 19/20 slicing from Listing 13
        skip_special_tokens=True)
    return reply.strip()

# 3. Setup Turn 1 manually (Sachin-bot speaks first)
turn_1_output = "Ayla! True class is Sachin Tendulkar.He's the god of cricket!"

print("--- TURN 1 (Sachin-bot) ---")
print(turn_1_output)
print("\n" + "="*50 + "\n")

current_message = turn_1_output

# 4. The 6-Turn Loop Chain
for turn in range(2, 7):
    if turn % 2 == 0:
        speaker = "Critic-Bot"
        selected_persona = persona_b
    else:
        speaker = "Sachin-bot"
        selected_persona = persona # Reuses your exact 'persona' variable from CELL 3

    print(f"--- TURN {turn} ({speaker}) ---")

    # Run the generation using your exact workshop function logic
    reply = workshop_debate_chat(selected_persona, f"Counter this directly: {current_message}")
    print(reply)
    print("\n" + "="*50 + "\n")

    # Feed this turn's reply as the next turn's input text
    current_message = reply


--- TURN 1 (Sachin-bot) ---
Ayla! True class is Sachin Tendulkar.He's the god of cricket!


--- TURN 2 (Critic-Bot) ---
(eyeing you warily) Ah, true class, you think you're the lord of the sea? You're not from the Caribbean, are you? Or is it something more?


--- TURN 3 (Sachin-bot) ---
(eyeing you warily) Ah, you're not from the Caribbean, are you? Or is it something more?

(eyeing you) I'm a professional cricketer, Sachin Tendulkar, and I don't think I can possibly understand your... perspective. I've seen your... my... experience. You're not from the Caribbean. You're from the land of the stars. The land of the stars!

(eyeing you) I don't know how you can talk about my experience with someone from the Caribbean. You're not from the Caribbean.


--- TURN 4 (Critic-Bot) ---
(eyeing you warily) Ah, I see. (tapping my finger on the table) And I'm not from the Caribbean. I'm from... (pausing, studying your reaction) ...the land of the stars. The land of the stars, where the stars shine